# User Guide
### Jeremy Ball, Omer Bowman, Richard Ngai


In [ ]:
# Import Libraries
import numpy as np
import time
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

# Connect to CoppeliaSim

The following code connects to an open session of CoppeliaSim. It will then load in the template MTB SCARA robot found in CoppeliaSim's model library.

In [ ]:
# Connect to CoppeliaSim
print('Connecting...')
client = RemoteAPIClient('localhost', 23000)
sim = client.require('sim')
print('Connected')
sim.setBoolParam(sim.boolparam_realtime_simulation, True)


# Load and configure the MTB SCARA

The user can choose to run this block to have the SCARA load in automatically. The following code
loads the MTB SCARA robot arm from CoppeliaSim's default library. The code also removes the default
scripts and box located within the heirarchy.

If the user is unable to run the code, they can manually place the MTB robot from the Model Browser
under robots>non-mobile>MTB robot. If manually added, the user must then delete the two scripts
under the MTB parent and Rectangle, which is the default box the model comes with.

In [ ]:
# Load robot
robot = sim.loadModel(r'C:\Program Files\CoppeliaRobotics\CoppeliaSimEdu\models\robots\non-mobile\MTB robot.ttm')
print('Robot loaded')

# Remove default script
script = sim.getScript(sim.scripttype_childscript, robot)
suction_script = sim.getScript(sim.scripttype_childscript, sim.getObject('/MTB/suctionPad'))
default_box = sim.getObject('/MTB/Rectangle')

if script != -1 and suction_script != -1:
    sim.removeObjects([script, suction_script, default_box])

# Assign Joints

The following code block assigns joints q1, q2, q3, and q4 based on the MTB's structure.

In [ ]:
# Assign MTB joints
q1 = sim.getObject('/MTB/axis')
q2 = sim.getObject('/MTB/axis/link/axis')
q3 = sim.getObject('/MTB/axis/link/axis/link/axis')
q4 = sim.getObject('/MTB/axis/link/axis/link/axis/axis')
q = [q1, q2, q3, q4]
print('Joints assigned')

# Get positions of the joints
o1 = np.array(sim.getObjectPosition(q1, sim.handle_world))  # Should be at about (0,0,z)
o2 = np.array(sim.getObjectPosition(q2, sim.handle_world))
o3 = np.array(sim.getObjectPosition(q3, sim.handle_world))



# Compute link lengths
l1 = o2[0]-o1[0]  # 0.467m
l2 = o3[0]-o2[0]  # 0.400m

# Get maximun suction pad actuation
tool = sim.getObject('/MTB/suctionPad/Link')
o4 = np.array(sim.getObjectPosition(tool, sim.handle_world))
q3_max = o4[2]
q3_up = 0.0

# Set up the Box class

Define a Box class that will be used to create boxes in the simulation.

In [ ]:
class Box:
    def __init__(self, sim, position: list, size: list=[0.1, 0.1, 0.1], velocity: list=[0.1, 0, 0], max_render: float=2.0):
        self.sim = sim
        self.size = size
        self.velocity = np.array(velocity)
        self.attached = False
        self.max_render = max_render

        self.handle = sim.createPrimitiveShape(
            sim.primitiveshape_cuboid,
            size,
            0
        )

        sim.setObjectPosition(self.handle, sim.handle_world, list(position))
        sim.setObjectInt32Param(self.handle, sim.shapeintparam_static, 1)
        sim.setShapeMass(self.handle, 0.1)
        sim.setShapeColor(self.handle, None, sim.colorcomponent_ambient_diffuse, [0.65, 0.65, 1])
    
    def update(self, dt):
        if self.handle == -1:
            return
        
        if not self.attached:
            pos = np.array(self.sim.getObjectPosition(self.handle, self.sim.handle_world))
            new_pos = pos + self.velocity * dt
            self.sim.setObjectPosition(self.handle, self.sim.handle_world, list(new_pos))

            # Automatically delete the box when it gets too far from the robot
            dist = np.sqrt(new_pos[0]**2 + new_pos[1]**2)
            if dist > self.max_render:
                self.delete()
    
    def attach(self, tool_handle):
        if self.handle == -1:
            return

        self.sim.setObjectParent(self.handle, tool_handle, True)
        self.attached = True

    def detach(self):
        if self.handle == -1:
            return

        self.sim.setObjectParent(self.handle, -1, True)
        self.attached = False
    
    def delete(self):
        if self.handle == -1:
            return
        try:
            # Detach if still parented
            parent = self.sim.getObjectParent(self.handle)
            if parent != -1:
                self.sim.setObjectParent(self.handle, -1, True)

            self.sim.removeObjects([self.handle])
            self.handle = -1
        
        except Exception as e:
            print(f"Failed to delete box: {e}")

# Robot Kinematic Functions

The following functions define the inverse kinematics used for the MTB robot.

In [ ]:
def inverse_kinematics(x: float, y: float, l1: float, l2: float) -> tuple:
    # Analytical SCARA IK using the paper's formulas.
    # θ2 = ± arccos([x^2 + y^2 - l1^2 - l2^2]/[2*l1*l2])
    # θ1 = atan2(y, x) - atan2(l2*sin(θ2), l1 + l2*cos(θ2))

    # Ensure target (x,y) is in robot's range
    r = np.sqrt(x**2 + y**2)
    if r < abs(l1 - l2) or r > (l1 + l2):
        raise ValueError(f'Target ({x:.3f}, {y:.3f}) is out of reach for l1={l1:.3f}, l2={l2:.3f}')

    cos_q2 = (x**2 + y**2 - l1**2 - l2**2) / (2*l1*l2)
    cos_q2 = np.clip(cos_q2, -1.0, 1.0)
    q2 = np.arccos(cos_q2)

    # Switch elbow configuration if on the "right" side of the robot
    if y < 0:
        q2 = -q2

    q1 = np.arctan2(y,x) - np.arctan2(l2*np.sin(q2), l1+l2*np.cos(q2))

    return q1, q2

def move_arm(sim, q: list, u: list):
    for i in range(len(q)):
        sim.setJointTargetPosition(q[i], u[i])

# Run Simulation

The following code runs the simulation in CoppeliaSim.

This simulation spawns a randomly-sized box within a given range of starting locations. The box then
"moves on a conveyor belt" which is done by manually updating the position of the box after each
timestep. The MTB arm performs inverse kinematics to position itself over the box and follow it,
lower the arm, attach the box, and then continue a bit with the "conveyor belt" speed before moving
the box to the other side.

On the other side of the robot is another "conveyor belt" which performs the same actions as before
but in reverse. Once the robot detaches the box and follows it for a bit, a new box is spawned. This
repeats 10 times.

In [ ]:
# Start the simulation
print('Starting simulation...')
sim.setStepping(True)
sim.startSimulation()
dt = sim.getSimulationTimeStep()


#~~~~~~~~~~~~~~~~~~~~~~~~~
# SIMULATION

boxes = []
num_boxes = 10
handled_count = 0


def spawn_box():
    box_y_start = np.random.uniform(-0.6, -0.3)
    box_size = []
    for _ in range(3):
        box_size.append(np.random.uniform(0.06, 0.11))

    box = Box(
        sim,
        position=[-0.03, box_y_start, box_size[2]/2],
        size=box_size,
        velocity=[0.1, 0.0, 0.0],
        max_render=2.0
    )
    boxes.append(box)
    return box


def update_all_boxes(exclude_box=None):
    for box in boxes[:]:
        if box is exclude_box:
            continue

        box.update(dt)

        if box.handle == -1:
            boxes.remove(box)


while handled_count < num_boxes:
    active_box = spawn_box()
    box_vel = active_box.velocity[:2].copy()
    q3_down = q3_max - active_box.size[2]

    # Follow the box
    for _ in range(50):
        update_all_boxes()
        box_pos = np.array(sim.getObjectPosition(active_box.handle, sim.handle_world))
        u1, u2 = inverse_kinematics(x=box_pos[0], y=box_pos[1], l1=l1, l2=l2)
        u = [u1, u2, q3_up, -(u1+u2)]
        move_arm(sim, q=q, u=u)
        sim.step()

    # Follow the box and lower the suction pad
    for _ in range(30):
        update_all_boxes()
        box_pos = np.array(sim.getObjectPosition(active_box.handle, sim.handle_world))
        u1, u2 = inverse_kinematics(x=box_pos[0], y=box_pos[1], l1=l1, l2=l2)
        u = [u1, u2, q3_down, -(u1+u2)]
        move_arm(sim, q=q, u=u)
        sim.step()

    # Attach the box to the tool
    active_box.attach(tool)
    xy_pos = sim.getObjectPosition(active_box.handle, sim.handle_world)[:2]

    # Raise the box but keep forward motion
    for _ in range(30):
        update_all_boxes()
        xy_pos[0] = xy_pos[0] + box_vel[0]*dt
        u1, u2 = inverse_kinematics(x=xy_pos[0], y=xy_pos[1], l1=l1, l2=l2)
        u = [u1, u2, q3_up, -(u1+u2)]
        move_arm(sim, q=q, u=u)
        sim.step()

    # Drop-off point
    xy_pos = [-0.3, 0.65]

    # Move to the drop off point and match conveyor velocity
    for _ in range(60):
        update_all_boxes()
        xy_pos[0] = xy_pos[0] + box_vel[0]*dt
        u1, u2 = inverse_kinematics(x=xy_pos[0], y=xy_pos[1], l1=l1, l2=l2)
        u = [u1, u2, q3_up, -(u1+u2)]
        move_arm(sim, q=q, u=u)
        sim.step()

    # Lower box at conveyor velocity
    for _ in range(30):
        update_all_boxes()
        xy_pos[0] = xy_pos[0] + box_vel[0]*dt
        u1, u2 = inverse_kinematics(x=xy_pos[0], y=xy_pos[1], l1=l1, l2=l2)
        u = [u1, u2, q3_down, -(u1+u2)]
        move_arm(sim, q=q, u=u)
        sim.step()

    # Detach box
    active_box.detach()
    handled_count += 1

    # Raise and follow for a bit
    for _ in range(20):
        update_all_boxes()
        box_pos = np.array(sim.getObjectPosition(active_box.handle, sim.handle_world))
        u1, u2 = inverse_kinematics(x=box_pos[0], y=box_pos[1], l1=l1, l2=l2)
        u = [u1, u2, q3_up, -(u1+u2)]
        move_arm(sim, q=q, u=u)
        sim.step()

#~~~~~~~~~~~~~~~~~~~~~~~~~


# Stop the simulation
sim.stopSimulation()
sim.setStepping(False)
print('Stopped')

# Remove MTB model
sim.removeModel(robot)